In [ ]:
code = 'B120G_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def b120_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om, seperate=False):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return '', None

        from_candle_close = True if method == 'CC' else False

        entry_time = start_dt
        _, _, _, _, ce_sl_price, ce_sl_time, ce_mtm_data = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        _, _, _, _, pe_sl_price, pe_sl_time, pe_mtm_data = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
        pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m
        ut_sl = ut_sl if str(ut_sl) == 'TTC' else float(ut_sl)
        
        if ce_sl_time < pe_sl_time:
            ut = 'PE'
            pe_mtm_data = pe_mtm_data[pe_mtm_data.index <= ce_sl_time]
            
            ut_sl_price = pe_price if str(ut_sl) == 'TTC' else None
            ut_open, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

            if ut_open:
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = pe_sl_price
                    _, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

        elif pe_sl_time < ce_sl_time:
            ut = 'CE'
            ce_mtm_data = ce_mtm_data[ce_mtm_data.index <= pe_sl_time]
            
            ut_sl_price = ce_price if str(ut_sl) == 'TTC' else None
            ut_open, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

            if ut_open:
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = ce_sl_price
                    _, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        else:
            ut = ''
            ut_sl_time, ut_mtm_data = '', pd.Series()
            
        ce_sl_time = '' if ce_sl_time == end_dt_1m else ce_sl_time
        pe_sl_time = '' if pe_sl_time == end_dt_1m else pe_sl_time
        ut_sl_time = '' if ut_sl_time == end_dt_1m else ut_sl_time
        
        if ut:
            ut_sl_time = end_dt if ut_sl_time == '' else ut_sl_time

        sl_times = [ce_sl_time, pe_sl_time, ut_sl_time]
        sl_times = [s for s in sl_times if s != '']
        b120_sl_time = max(sl_times) if sl_times else ''
        b120_sl_time = '' if b120_sl_time == end_dt else b120_sl_time

        if seperate:
            return b120_sl_time, ce_mtm_data, pe_mtm_data, ut, ut_mtm_data
        else:
            ce_mtm_data = set_pm_time_index(ce_mtm_data, time_index)
            pe_mtm_data = set_pm_time_index(pe_mtm_data, time_index)
            
            if ut:
                ut_mtm_data = set_pm_time_index(ut_mtm_data, time_index)
                return b120_sl_time, ce_mtm_data+pe_mtm_data+ut_mtm_data
            else:
                return b120_sl_time, ce_mtm_data+pe_mtm_data
    
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, ut_sl, om])
        return '', None

def b120G_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om, start_time2, end_time2, sl2, ut_sl2, om2, seperate=False):
    try:
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        b120_1_sl_time, b120_1_mtm_data = b120_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om)
        if b120_1_mtm_data is None:
            return None
        
        b120_2_sl_time, b120_2_mtm_data = b120_per_minute_mtm(bt, start_time2, end_time2, orderside, method, sl2, ut_sl2, om2)
        if b120_2_mtm_data is None:
            b120_2_mtm_data = set_pm_time_index(pd.Series(), time_index)

        if seperate:
            return b120_1_mtm_data, b120_2_mtm_data
        else:
            return b120_1_mtm_data + b120_2_mtm_data

    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, ut_sl, om, start_time2, end_time2, sl2, ut_sl2, om2])
        return

In [ ]:
def b120_RE_PSL(bt, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, ut_sl, om, start_time2, end_time2, last_trade_time2, trade_interval2, sl2, ut_sl2, om2):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        start_dt2 = datetime.datetime.combine(bt.current_date, start_time2)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        last_trade_dt = datetime.datetime.combine(bt.current_date, last_trade_time)
        last_trade_dt2 = datetime.datetime.combine(bt.current_date, last_trade_time2)

        entry_time = start_dt
        time_range = pd.date_range(start_dt, last_trade_dt, freq=trade_interval.lower()).time
        time_range2 = pd.date_range(start_dt2, last_trade_dt2, freq=trade_interval2.lower()).time
        
        if len(time_range) != len(time_range2): raise Exception("B120G TimeMismatch !!!")

        per_minute_trades = [b120G_per_minute_mtm(bt, re_time, end_time, orderside, method, sl, ut_sl, om, re_time2, end_time2, sl2, ut_sl2, om2) for re_time, re_time2 in zip(time_range, time_range2)]
        per_minute_trades = [t for t in per_minute_trades if t is not None]
        per_minute_mtm = np.sum(per_minute_trades, axis=0)
        
        if only_mtm:
            mtm_time_list = list(per_minute_mtm)
            mtm_time_list += [mtm_time_list[-1]]
        else:
            total_minutes = len(time_range)
            per_minute_margin = int(fund/total_minutes)
            shares_per_minute = int(per_minute_margin/margin_per_share)
            per_minute_mtm_total = per_minute_mtm * shares_per_minute

            mtm_dict ={}
            for mtm_percent, check_mtm in check_mtms.items():

                if check_mtm > 0:
                    condition = np.where(per_minute_mtm_total > check_mtm)[0]
                else:
                    condition = np.where(per_minute_mtm_total < check_mtm)[0]

                try:
                    idx = condition[0]
                    time = time_index[idx]
                    mtm = per_minute_mtm_total[idx+1]
                except:
                    time, mtm = '', per_minute_mtm_total[-1]

                mtm_dict[mtm_percent] = (time, mtm)

            mtm_time_list = [fund] + [item for value in mtm_dict.values() for item in value]
       
        return [code, bt.index, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, cv(ut_sl), om, start_time2, end_time2, last_trade_time2, trade_interval2, sl2, cv(ut_sl2), om2, bt.current_date.date(), bt.current_date.day_name(), dte, entry_time.time(), future_price] + mtm_time_list
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, ut_sl, om, start_time2, end_time2, last_trade_time2, trade_interval2, sl2, ut_sl2, om2])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_LastTradeTime/P_TradeInterval/P_OrderSide/P_Method/P_SL/P_UTSL/P_OM/P_StartTime2/P_EndTime2/P_LastTradeTime2/P_TradeInterval2/P_SL2/P_UTSL2/P_OM2/Date/Day/DTE/Entry.Time/Future/Fund')
            
            only_mtm = False
            if only_mtm:
                log_time_col = get_pm_time_index(datetime.datetime.now()).time
                log_cols += '/'.join(map(str, log_time_col))
                log_cols += '/Final.PNL'
            else:
                notinal_value = 12
                fund = 100_00_00_00
                mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -100, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
                log_cols += 'Fund'
                for mtmp in mtm_percent_stop:
                    log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'

            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    time_index = get_pm_time_index(bt.current_date)
                    future_price = bt.future_data['close'].iloc[0]
                    
                    if not only_mtm:
                        margin_per_share = future_price * (notinal_value / 100)
                        check_mtms = {mtm_percent: fund * mtm_percent / 100 for mtm_percent in mtm_percent_stop}
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [b120_RE_PSL(bt, row.entry_time, row.exit_time, row.last_trade_time, row.trade_interval, row.orderside, row.method, row.sl, row.ut_sl, row.om, row.entry_time2, row.exit_time2, row.last_trade_time2, row.trade_interval2, row.sl2, row.ut_sl2, row.om2) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))